# (Optional) Deploy Legacy Agent (legacy)

비교 데모용 `fhir_agent_legacy` AgentCore Runtime을 배포합니다.

```
React UI (:4000) → server.py (:4001) → AgentCore Runtime (with_metadata)    → fhir-mcp-server-gw       → data namespace
React UI (:6000) → server.py (:6001) → AgentCore Runtime (legacy)  → fhir-mcp-legacy-gw → data_legacy namespace
```

## Step 1: Configuration

In [ ]:
import boto3, json, time, os, zipfile, io, uuid

REGION = boto3.session.Session().region_name or os.environ.get("AWS_DEFAULT_REGION") or os.environ.get("AWS_REGION")
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
S3_BUCKET = f"fhir-data-{ACCOUNT_ID}-{REGION}"

with open("legacy_connection_info.json") as f:
    comp_conn = json.load(f)

iam_client = boto3.client("iam")
s3_client = boto3.client("s3", region_name=REGION)
agentcore = boto3.client("bedrock-agentcore-control", region_name=REGION)

AGENT_NAME = "fhir_agent_legacy"
GATEWAY_URL = comp_conn["legacy"]["gateway_url"]
AUTH = comp_conn["auth"]
TARGET_NAME = comp_conn["legacy"]["target_name"]

print(f"Account: {ACCOUNT_ID}, Region: {REGION}")
print(f"Gateway URL: {GATEWAY_URL}")


## Step 2: Enable CloudWatch Transaction Search

In [ ]:
logs_client = boto3.client("logs", region_name=REGION)
try:
    policy_doc = json.dumps({
        "Version": "2012-10-17",
        "Statement": [{"Sid": "TransactionSearchXRayAccess", "Effect": "Allow",
            "Principal": {"Service": "xray.amazonaws.com"}, "Action": "logs:PutLogEvents",
            "Resource": [f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:aws/spans:*",
                         f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:/aws/application-signals/data:*"],
            "Condition": {"ArnLike": {"aws:SourceArn": f"arn:aws:xray:{REGION}:{ACCOUNT_ID}:*"},
                          "StringEquals": {"aws:SourceAccount": ACCOUNT_ID}}}]
    })
    logs_client.put_resource_policy(policyName="AgentCoreTransactionSearch", policyDocument=policy_doc)
    print("CloudWatch policy set")
except Exception as e:
    print(f"Skipped: {e}")


## Step 3: Create AgentCore Memory

In [ ]:
mem = agentcore.create_memory(
    name=f"fhir_legacy_mem_{uuid.uuid4().hex[:8]}",
    eventExpiryDuration=30,
    memoryStrategies=[
        {"userPreferenceMemoryStrategy": {"name": "user_prefs", "namespaces": ["/users/preferences/"]}},
        {"semanticMemoryStrategy": {"name": "medical_facts", "namespaces": ["/users/facts/"]}}
    ]
)
MEMORY_ID = mem["memory"]["id"]
print(f"Memory ID: {MEMORY_ID}")

print("Waiting for memory ACTIVE...")
for _ in range(60):
    if agentcore.get_memory(memoryId=MEMORY_ID)["memory"]["status"] == "ACTIVE":
        print("Memory ACTIVE!")
        break
    time.sleep(5)


## Step 4: Create IAM Role for AgentCore Runtime

In [ ]:
RUNTIME_ROLE_NAME = "fhir-legacy-runtime-role"

trust_policy = {"Version": "2012-10-17", "Statement": [
    {"Effect": "Allow", "Principal": {"Service": "bedrock-agentcore.amazonaws.com"}, "Action": "sts:AssumeRole"}
]}

try:
    iam_client.create_role(RoleName=RUNTIME_ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy))
    print(f"Created role: {RUNTIME_ROLE_NAME}")
except iam_client.exceptions.EntityAlreadyExistsException:
    print(f"Role exists: {RUNTIME_ROLE_NAME}")

RUNTIME_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{RUNTIME_ROLE_NAME}"

iam_client.put_role_policy(RoleName=RUNTIME_ROLE_NAME, PolicyName="AgentCoreRuntimePolicy", PolicyDocument=json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"], "Resource": "*"},
        {"Effect": "Allow", "Action": ["bedrock-agentcore:*"], "Resource": "*"},
        {"Effect": "Allow", "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"], "Resource": "*"},
        {"Effect": "Allow", "Action": ["xray:PutTraceSegments", "xray:PutTelemetryRecords"], "Resource": "*"},
        {"Effect": "Allow", "Action": ["ecr:GetAuthorizationToken", "ecr:BatchGetImage", "ecr:GetDownloadUrlForLayer"], "Resource": "*"},
    ]
}))
print(f"Role ARN: {RUNTIME_ROLE_ARN}")
time.sleep(10)


## Step 5: Build & Push Container Image

In [ ]:
ecr_client = boto3.client("ecr", region_name=REGION)
codebuild_client = boto3.client("codebuild", region_name=REGION)

ECR_REPO_NAME = "fhir-legacy-agent"
try:
    ecr_client.create_repository(repositoryName=ECR_REPO_NAME)
    print(f"ECR repo created: {ECR_REPO_NAME}")
except ecr_client.exceptions.RepositoryAlreadyExistsException:
    print(f"ECR repo exists: {ECR_REPO_NAME}")

ECR_URI = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{ECR_REPO_NAME}:latest"

# Upload agent source
agent_dir = "../agent"
zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in ["medical_agent.py", "system_prompt.md", "system_prompt_legacy.md", "requirements.txt", "Dockerfile"]:
        filepath = os.path.join(agent_dir, fname)
        if os.path.exists(filepath):
            zf.write(filepath, fname)
    buildspec = """version: 0.2
phases:
  pre_build:
    commands:
      - aws ecr get-login-password --region $AWS_DEFAULT_REGION | docker login --username AWS --password-stdin $ECR_REGISTRY
  build:
    commands:
      - docker build -t $ECR_REGISTRY/$ECR_REPO:latest .
  post_build:
    commands:
      - docker push $ECR_REGISTRY/$ECR_REPO:latest
"""
    zf.writestr("buildspec.yml", buildspec)
zip_buffer.seek(0)
s3_key = "build/legacy-agent-source.zip"
s3_client.put_object(Bucket=S3_BUCKET, Key=s3_key, Body=zip_buffer.getvalue())
print(f"Source uploaded to s3://{S3_BUCKET}/{s3_key}")

# CodeBuild service role
CB_ROLE_NAME = "fhir-legacy-codebuild-role"
cb_trust = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "codebuild.amazonaws.com"}, "Action": "sts:AssumeRole"}]
}
try:
    iam_client.create_role(RoleName=CB_ROLE_NAME, AssumeRolePolicyDocument=json.dumps(cb_trust))
except iam_client.exceptions.EntityAlreadyExistsException:
    pass
iam_client.put_role_policy(RoleName=CB_ROLE_NAME, PolicyName="CodeBuildPolicy", PolicyDocument=json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": ["ecr:*"], "Resource": "*"},
        {"Effect": "Allow", "Action": ["ecr:GetAuthorizationToken"], "Resource": "*"},
        {"Effect": "Allow", "Action": ["s3:GetObject", "s3:GetBucketLocation"], "Resource": [f"arn:aws:s3:::{S3_BUCKET}", f"arn:aws:s3:::{S3_BUCKET}/*"]},
        {"Effect": "Allow", "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"], "Resource": "*"}
    ]
}))
CB_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{CB_ROLE_NAME}"
print(f"CodeBuild role ready: {CB_ROLE_ARN}")
time.sleep(10)

# Create/update CodeBuild project
CB_PROJECT = "fhir-legacy-agent-build"
ECR_REGISTRY = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com"
project_config = dict(
    name=CB_PROJECT,
    source={"type": "S3", "location": f"{S3_BUCKET}/{s3_key}"},
    artifacts={"type": "NO_ARTIFACTS"},
    environment={
        "type": "ARM_CONTAINER",
        "image": "aws/codebuild/amazonlinux2-aarch64-standard:3.0",
        "computeType": "BUILD_GENERAL1_SMALL",
        "privilegedMode": True,
        "environmentVariables": [
            {"name": "ECR_REGISTRY", "value": ECR_REGISTRY},
            {"name": "ECR_REPO", "value": ECR_REPO_NAME},
        ]
    },
    serviceRole=CB_ROLE_ARN,
)
try:
    codebuild_client.create_project(**project_config)
    print(f"CodeBuild project created: {CB_PROJECT}")
except codebuild_client.exceptions.ResourceAlreadyExistsException:
    codebuild_client.update_project(**project_config)
    print(f"CodeBuild project updated: {CB_PROJECT}")

# Start build
build = codebuild_client.start_build(projectName=CB_PROJECT)
BUILD_ID = build["build"]["id"]
print(f"Build started: {BUILD_ID}")
print("Waiting (~2-3 min)...")
for i in range(60):
    b = codebuild_client.batch_get_builds(ids=[BUILD_ID])["builds"][0]
    bs = b["buildStatus"]
    if bs == "SUCCEEDED":
        print(f"Build SUCCEEDED! Image: {ECR_URI}")
        break
    elif bs in ("FAILED", "FAULT", "STOPPED"):
        print(f"Build {bs}")
        for p in b.get("phases", []):
            if p.get("phaseStatus") == "FAILED":
                print(f"  Failed phase: {p['phaseType']} - {p.get('contexts', [{}])[0].get('message', '')}")
        break
    print(f"  {bs} ({i*10}s)", end="\r")
    time.sleep(10)


## Step 6: Deploy AgentCore Runtime

In [ ]:
env_vars = {
    "AWS_REGION": REGION,
    "BEDROCK_AGENTCORE_MEMORY_ID": MEMORY_ID,
    "MODEL_ID": "us.anthropic.claude-sonnet-4-6",
    "GATEWAY_MCP_URL": GATEWAY_URL,
    "GATEWAY_CLIENT_ID": AUTH["client_id"],
    "GATEWAY_CLIENT_SECRET": AUTH["client_secret"],
    "GATEWAY_TOKEN_URL": AUTH["token_url"],
    "GATEWAY_SCOPE": "fhir-mcp/tools",
    "ENABLED_TOOLS": "list_tables,get_table_schema,get_table_relationships,run_custom_query",
    "SYSTEM_PROMPT_FILE": "system_prompt_legacy.md",
    "GATEWAY_TOOL_PREFIX": f"{TARGET_NAME}___",
}

try:
    resp = agentcore.create_agent_runtime(
        agentRuntimeName=AGENT_NAME,
        agentRuntimeArtifact={"containerConfiguration": {"containerUri": ECR_URI}},
        roleArn=RUNTIME_ROLE_ARN,
        networkConfiguration={"networkMode": "PUBLIC"},
        environmentVariables=env_vars,
    )
    AGENT_ID = resp["agentRuntimeId"]
    AGENT_ARN = resp["agentRuntimeArn"]
    print(f"Created: {AGENT_ARN}")
except agentcore.exceptions.ConflictException:
    runtimes = agentcore.list_agent_runtimes()["agentRuntimes"]
    agent = next(r for r in runtimes if r["agentRuntimeName"] == AGENT_NAME)
    AGENT_ID = agent["agentRuntimeId"]
    AGENT_ARN = agent["agentRuntimeArn"]
    agentcore.update_agent_runtime(
        agentRuntimeId=AGENT_ID,
        agentRuntimeArtifact={"containerConfiguration": {"containerUri": ECR_URI}},
        environmentVariables=env_vars,
    )
    print(f"Updated: {AGENT_ARN}")

print(f"Agent ID: {AGENT_ID}")


## Step 7: Wait for Runtime READY & Create Endpoint

In [ ]:
print(f"Waiting for {AGENT_NAME} READY...")
for i in range(60):
    status = agentcore.get_agent_runtime(agentRuntimeId=AGENT_ID)["status"]
    if status == "READY":
        print(f"READY! (~{i*10}s)")
        break
    elif "FAILED" in status:
        print(f"FAILED: {status}")
        break
    print(f"  {status} ({i*10}s)", end="\r")
    time.sleep(10)

version = str(agentcore.get_agent_runtime(agentRuntimeId=AGENT_ID).get("agentRuntimeVersion", "1"))
try:
    agentcore.create_agent_runtime_endpoint(agentRuntimeId=AGENT_ID, name="DEFAULT", agentRuntimeVersion=version)
    print("Endpoint created")
except agentcore.exceptions.ConflictException:
    print("Endpoint exists")

print("Waiting for endpoint READY...")
for i in range(60):
    ep = agentcore.get_agent_runtime_endpoint(agentRuntimeId=AGENT_ID, endpointName="DEFAULT")
    if ep["status"] == "READY":
        print(f"Endpoint READY! (~{i*10}s)")
        break
    elif "FAILED" in ep["status"]:
        print(f"Endpoint FAILED")
        break
    time.sleep(10)


## Step 8: Save Agent Info

In [ ]:
agent_info = {
    "fhir_agent_legacy": {"arn": AGENT_ARN, "id": AGENT_ID, "memory_id": MEMORY_ID},
    "region": REGION,
}
with open("legacy_agent_info.json", "w") as f:
    json.dump(agent_info, f, indent=2)
print("Saved to legacy_agent_info.json")
print(json.dumps(agent_info, indent=2))


## ✅ Done!

`fhir_agent_legacy` AgentCore Runtime이 배포되었습니다.

**React UI 실행:**
```bash
# 터미널 1 — With Metadata (메인)
./run_react_app.sh          # http://<EC2-IP>:3000

# 터미널 2 — Legacy (비교)
./run_react_app_legacy.sh  # http://<EC2-IP>:4000
```

두 브라우저 탭을 나란히 열고 동일한 질문으로 AI-Ready Data Lake의 차이를 비교해보세요.